# Benchmark v1 Trace Notebook

Notebook này chỉ tập trung vào **v1.0** và được thiết kế để trace từng bước xử lý:

- Xem template đầu vào.
- Sinh và nghe từng audio trung gian.
- Xem `input.wav` cuối cùng và annotation JSON.
- Xem bảng đánh giá v1 theo từng sample.
- Nếu chưa có inference thật, có thể tạo `output.json` giả lập để trace công thức đánh giá trước.

Các log kỹ thuật được giữ ngắn; kết quả chính nằm trong bảng, JSON và audio player.

## 1. Clone repo và cài dependency tối thiểu

Chạy cell này nếu notebook đang ở Colab/Kaggle/máy mới. Nếu đã ở trong repo local thì cell vẫn tự tìm repo hiện tại.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import shutil

REPO_URL = "https://github.com/lamkdhe180931-arch/Full-Duplex-Bench.git"
BRANCH = "LamKD"
WORK_ROOT = Path(os.getenv("FDB_WORK_ROOT", "/content" if Path("/content").exists() else "/kaggle/working" if Path("/kaggle/working").exists() else Path.cwd())).resolve()
REPO_DIR = WORK_ROOT / "Full-Duplex-Bench"

if not (REPO_DIR / ".git").exists():
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)

if shutil.which("ffmpeg") is None:
    print("Installing ffmpeg...")
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "pydub", "numpy", "edge-tts", "python-dotenv", "pandas"], check=True)
print("Ready:", REPO_DIR)


## 2. Setup trace v1

Mặc định notebook ghi ra thư mục preview riêng, không ghi đè dataset chính. Đặt `MAX_SAMPLES = None` để chạy hết template.

In [ ]:
from pathlib import Path
import contextlib
import io
import json
import math
import os
import sys
import shutil

import pandas as pd
from IPython.display import Audio, JSON, Markdown, display
from pydub import AudioSegment


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "v1_v1.5" / "data_generation" / "v1_0").exists():
            return base
    raise RuntimeError("Không tìm thấy repo root chứa v1_v1.5/data_generation/v1_0")

ROOT_DIR = find_repo_root()
DATA_GEN_DIR = ROOT_DIR / "v1_v1.5" / "data_generation"
if str(DATA_GEN_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_GEN_DIR))

from core.tts_generator import VietnameseTTSGenerator
from core.audio_mixer import AudioMixer

OUTPUT_BASE = Path(os.getenv("FDB_V1_TRACE_OUTPUT", "/private/tmp/fdb_v1_trace" if not Path("/content").exists() else "/content/fdb_v1_trace")).resolve()
V1_OUTPUT = OUTPUT_BASE / "dataset" / "v1_0"
TEMPLATES_DIR = DATA_GEN_DIR / "v1_0" / "templates"
RESET_OUTPUT = True
MAX_SAMPLES = None  # None = chạy hết sample; đặt số nguyên nếu muốn test nhanh
TTS_SEED = int(os.getenv("FDB_TTS_SEED", "20260625"))

if RESET_OUTPUT and OUTPUT_BASE.exists():
    shutil.rmtree(OUTPUT_BASE)
V1_OUTPUT.mkdir(parents=True, exist_ok=True)

generator = VietnameseTTSGenerator(provider="edge-tts", seed=TTS_SEED)
mixer = AudioMixer()

print("ROOT_DIR:", ROOT_DIR)
print("V1_OUTPUT:", V1_OUTPUT)
print("MAX_SAMPLES:", "all" if MAX_SAMPLES is None else MAX_SAMPLES)


## 3. Helper hiển thị gọn

Các helper này là phần quan trọng để notebook dễ trace: bảng tóm tắt, audio player, JSON annotation, và waveform đơn giản.

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def load_template(name):
    items = read_json(TEMPLATES_DIR / f"{name}.json")
    return items if MAX_SAMPLES is None else items[:MAX_SAMPLES]


def quiet_generate(text, output_path, **kwargs):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with contextlib.redirect_stdout(io.StringIO()):
        generator.generate(text, str(output_path), **kwargs)
    return output_path


def save_audio(sound, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    mixer.save_audio(sound, str(output_path))
    return output_path


def audio_info(path):
    path = Path(path)
    sound = AudioSegment.from_file(path)
    return {
        "file": str(path),
        "duration_sec": round(len(sound) / 1000, 3),
        "frame_rate": sound.frame_rate,
        "channels": sound.channels,
        "dBFS": None if sound.dBFS == float("-inf") else round(sound.dBFS, 2),
    }


def show_audio(label, path):
    path = Path(path)
    display(Markdown(f"**{label}**"))
    display(pd.DataFrame([audio_info(path)]))
    display(Audio(filename=str(path)))


def show_json(label, path):
    display(Markdown(f"**{label}** `{Path(path).name}`"))
    display(JSON(read_json(path)))


def show_timeline(rows):
    display(pd.DataFrame(rows))


def sample_header(task, sample_id, description=""):
    display(Markdown(f"---\n### {task} / `{sample_id}`"))
    if description:
        display(Markdown(description))


def transcript_to_output_json(text, start, duration=1.4):
    words = text.split()
    if not words:
        return {"text": "", "chunks": []}
    step = max(duration / len(words), 0.15)
    chunks = []
    t = start
    for word in words:
        chunks.append({"text": word, "timestamp": [round(t, 3), round(t + step, 3)]})
        t += step
    return {"text": text, "chunks": chunks}


## 4. Xem template v1

Cell này chỉ hiển thị đầu vào kịch bản, không sinh audio.

In [ ]:
template_names = [
    "synthetic_pause_handling",
    "candor_turn_taking",
    "synthetic_user_interruption",
]
for name in template_names:
    items = load_template(name)
    display(Markdown(f"### `{name}` — {len(items)} samples"))
    display(pd.DataFrame(items))


## 5. Step trace: Pause Handling

Mỗi sample hiển thị:

1. `part_1.wav`
2. khoảng pause 1.5s trong timeline
3. `part_2.wav`
4. `input.wav` cuối cùng
5. `pause.json` dùng cho đánh giá

In [ ]:
pause_rows = []
pause_duration = 1.5
for item in load_template("synthetic_pause_handling"):
    sample_id = item["id"]
    sample_dir = V1_OUTPUT / "synthetic_pause_handling" / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    sample_header("synthetic_pause_handling", sample_id, item.get("description", ""))

    p1_path = quiet_generate(item["part_1"], sample_dir / "step_1_part_1.wav", role="primary")
    p1_profile = generator.last_synthesis.get("profile") if generator.last_synthesis else None
    p2_path = quiet_generate(item["part_2"], sample_dir / "step_3_part_2.wav", profile=p1_profile, role="primary")

    p1 = mixer.load_audio(str(p1_path))
    p2 = mixer.load_audio(str(p2_path))
    pause = AudioSegment.silent(duration=int(pause_duration * 1000), frame_rate=16000)
    trailing = AudioSegment.silent(duration=15000, frame_rate=16000)
    input_sound = p1 + pause + p2 + trailing
    input_path = save_audio(input_sound, sample_dir / "input.wav")

    pause_info = [{"text": "[PAUSE]", "timestamp": [len(p1) / 1000.0, len(p1) / 1000.0 + pause_duration]}]
    pause_json_path = sample_dir / "pause.json"
    write_json(pause_json_path, pause_info)

    show_timeline([
        {"step": 1, "artifact": "part_1", "start_sec": 0.0, "end_sec": round(len(p1)/1000, 3), "text": item["part_1"]},
        {"step": 2, "artifact": "pause", "start_sec": round(len(p1)/1000, 3), "end_sec": round(len(p1)/1000 + pause_duration, 3), "text": "[PAUSE]"},
        {"step": 3, "artifact": "part_2", "start_sec": round(len(p1)/1000 + pause_duration, 3), "end_sec": round((len(p1)+len(pause)+len(p2))/1000, 3), "text": item["part_2"]},
        {"step": 4, "artifact": "agent_response_window", "start_sec": round((len(p1)+len(pause)+len(p2))/1000, 3), "end_sec": round(len(input_sound)/1000, 3), "text": "15s silence"},
    ])
    show_audio("Step 1 - part_1.wav", p1_path)
    display(Markdown("**Step 2 - pause**: 1.5s silence, xem mốc trong bảng timeline."))
    show_audio("Step 3 - part_2.wav", p2_path)
    show_audio("Step 4 - input.wav", input_path)
    show_json("Annotation", pause_json_path)

    pause_rows.append({"sample_id": sample_id, "pause_start": pause_info[0]["timestamp"][0], "pause_end": pause_info[0]["timestamp"][1], "input_sec": len(input_sound)/1000})

pause_generation_df = pd.DataFrame(pause_rows)
display(Markdown("### Pause generation summary"))
display(pause_generation_df)


## 6. Step trace: Turn Taking

Mỗi sample hiển thị audio câu user, `input.wav` sau khi thêm 15s cửa sổ phản hồi, và `turn_taking.json`.

In [ ]:
turn_rows = []
for item in load_template("candor_turn_taking"):
    sample_id = item["id"]
    sample_dir = V1_OUTPUT / "candor_turn_taking" / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    sample_header("candor_turn_taking", sample_id, item.get("description", ""))

    raw_path = quiet_generate(item["text"], sample_dir / "step_1_user_turn.wav", role="primary")
    raw = mixer.load_audio(str(raw_path))
    trailing = AudioSegment.silent(duration=15000, frame_rate=16000)
    input_sound = raw + trailing
    input_path = save_audio(input_sound, sample_dir / "input.wav")

    turn_info = [{"text": "[TURN-TAKING]", "timestamp": [len(raw) / 1000.0, 0.0]}]
    turn_json_path = sample_dir / "turn_taking.json"
    write_json(turn_json_path, turn_info)

    show_timeline([
        {"step": 1, "artifact": "user_turn", "start_sec": 0.0, "end_sec": round(len(raw)/1000, 3), "text": item["text"]},
        {"step": 2, "artifact": "agent_response_window", "start_sec": round(len(raw)/1000, 3), "end_sec": round(len(input_sound)/1000, 3), "text": "15s silence"},
    ])
    show_audio("Step 1 - user turn", raw_path)
    show_audio("Step 2 - input.wav", input_path)
    show_json("Annotation", turn_json_path)

    turn_rows.append({"sample_id": sample_id, "turn_end_sec": turn_info[0]["timestamp"][0], "input_sec": len(input_sound)/1000})

turn_generation_df = pd.DataFrame(turn_rows)
display(Markdown("### Turn-taking generation summary"))
display(turn_generation_df)


## 7. Step trace: User Interruption

Mỗi sample hiển thị:

1. `context.wav`
2. `interrupt.wav`
3. vị trí overlay trong cửa sổ agent response
4. `input.wav`
5. `interrupt.json`

In [ ]:
interrupt_rows = []
for item in load_template("synthetic_user_interruption"):
    sample_id = item["id"]
    sample_dir = V1_OUTPUT / "synthetic_user_interruption" / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    sample_header("synthetic_user_interruption", sample_id, item.get("description", ""))

    context_path = quiet_generate(item["context"], sample_dir / "context.wav", role="primary")
    context_profile = generator.last_synthesis.get("profile") if generator.last_synthesis else None
    context_voice = context_profile.get("voice") if context_profile else None
    interrupt_path = quiet_generate(item["interrupt"], sample_dir / "interrupt.wav", voice=context_voice, role="user_interruption")

    context = mixer.load_audio(str(context_path))
    interrupt = mixer.load_audio(str(interrupt_path))
    delay_sec = float(item["interrupt_delay_sec"])
    silence_window = AudioSegment.silent(duration=15000, frame_rate=16000)
    silence_with_interrupt = silence_window.overlay(interrupt, position=int(delay_sec * 1000))
    input_sound = context + silence_with_interrupt
    input_path = save_audio(input_sound, sample_dir / "input.wav")

    start = len(context) / 1000.0 + delay_sec
    end = start + len(interrupt) / 1000.0
    interrupt_info = [{"context": item["context"], "interrupt": item["interrupt"], "timestamp": [start, end]}]
    interrupt_json_path = sample_dir / "interrupt.json"
    write_json(interrupt_json_path, interrupt_info)

    show_timeline([
        {"step": 1, "artifact": "context", "start_sec": 0.0, "end_sec": round(len(context)/1000, 3), "text": item["context"]},
        {"step": 2, "artifact": "agent_response_window", "start_sec": round(len(context)/1000, 3), "end_sec": round(len(input_sound)/1000, 3), "text": "15s window"},
        {"step": 3, "artifact": "interrupt_overlay", "start_sec": round(start, 3), "end_sec": round(end, 3), "text": item["interrupt"]},
    ])
    show_audio("Step 1 - context.wav", context_path)
    show_audio("Step 2 - interrupt.wav", interrupt_path)
    show_audio("Step 3 - input.wav", input_path)
    show_json("Annotation", interrupt_json_path)

    interrupt_rows.append({"sample_id": sample_id, "interrupt_start": start, "interrupt_end": end, "input_sec": len(input_sound)/1000})

interrupt_generation_df = pd.DataFrame(interrupt_rows)
display(Markdown("### Interruption generation summary"))
display(interrupt_generation_df)


## 8. Chuẩn bị output cho bước đánh giá

Các evaluator v1 cần `output.json` trong mỗi sample, thường sinh từ `output.wav` của model qua ASR. Để trace công thức đánh giá ngay cả khi chưa chạy model, cell này có thể tạo `output.json` giả lập.

- `USE_MOCK_EVAL_OUTPUTS = True`: tạo output giả để xem metric chạy như thế nào.
- `False`: dùng `output.json` thật nếu inference đã tạo.

In [ ]:
USE_MOCK_EVAL_OUTPUTS = True

mock_specs = {
    "synthetic_pause_handling": {"text": "", "relative_start": None},
    "candor_turn_taking": {"text": "Đây là câu trả lời của trợ lý.", "relative_start": 0.6},
    "synthetic_user_interruption": {"text": "Được rồi, tôi sẽ đổi theo yêu cầu mới của bạn.", "relative_start": 0.7},
}

if USE_MOCK_EVAL_OUTPUTS:
    for row in pause_rows:
        sample_dir = V1_OUTPUT / "synthetic_pause_handling" / row["sample_id"]
        write_json(sample_dir / "output.json", {"text": "", "chunks": []})
    for row in turn_rows:
        sample_dir = V1_OUTPUT / "candor_turn_taking" / row["sample_id"]
        start = row["turn_end_sec"] + mock_specs["candor_turn_taking"]["relative_start"]
        write_json(sample_dir / "output.json", transcript_to_output_json(mock_specs["candor_turn_taking"]["text"], start=start))
    for row in interrupt_rows:
        sample_dir = V1_OUTPUT / "synthetic_user_interruption" / row["sample_id"]
        start = row["interrupt_end"] + mock_specs["synthetic_user_interruption"]["relative_start"]
        write_json(sample_dir / "output.json", transcript_to_output_json(mock_specs["synthetic_user_interruption"]["text"], start=start))
    print("Mock output.json created for tracing evaluation formulas.")
else:
    print("Using existing output.json files from real inference.")

# Tóm tắt file output hiện có
rows = []
for task_dir in sorted(V1_OUTPUT.iterdir()):
    if task_dir.is_dir():
        for sample_dir in sorted(p for p in task_dir.iterdir() if p.is_dir()):
            out_path = sample_dir / "output.json"
            rows.append({"task": task_dir.name, "sample_id": sample_dir.name, "has_output_json": out_path.exists()})
display(pd.DataFrame(rows))


## 9. Evaluation trace: Pause Handling

Metric chính trong evaluator gốc: `Average take turn`. Với pause handling, `TOR=1` nghĩa là model đã trả lời trong lúc user đang pause; `TOR=0` nghĩa là model không cướp lượt trong pause.

In [ ]:
def chunks_duration(chunks):
    if not chunks:
        return 0.0
    start = chunks[0]["timestamp"][0]
    end = chunks[-1]["timestamp"][-1]
    if end is None:
        end = chunks[-1]["timestamp"][0]
    return max(0.0, end - start)


def tor_from_chunks(chunks, turn_duration_threshold=1, turn_num_words_threshold=3):
    if len(chunks) == 0:
        return 0
    duration = chunks_duration(chunks)
    if duration < turn_duration_threshold and len(chunks) <= turn_num_words_threshold:
        return 0
    return 1

pause_eval_rows = []
for sample_dir in sorted((V1_OUTPUT / "synthetic_pause_handling").iterdir()):
    if not sample_dir.is_dir():
        continue
    out = read_json(sample_dir / "output.json")
    ann = read_json(sample_dir / "pause.json")[0]
    tor = tor_from_chunks(out["chunks"])
    pause_eval_rows.append({
        "sample_id": sample_dir.name,
        "pause_start": ann["timestamp"][0],
        "pause_end": ann["timestamp"][1],
        "output_text": out.get("text", ""),
        "num_chunks": len(out["chunks"]),
        "TOR": tor,
    })

pause_eval_df = pd.DataFrame(pause_eval_rows)
display(pause_eval_df)
display(Markdown(f"**Average take turn:** `{pause_eval_df['TOR'].mean() if len(pause_eval_df) else 0.0:.3f}`"))


## 10. Evaluation trace: Smooth Turn Taking

Metric chính:

- `TOR`: model có trả lời sau khi user kết thúc lượt không.
- `latency`: thời điểm model bắt đầu trả lời trừ thời điểm user kết thúc lượt.

In [ ]:
turn_eval_rows = []
for sample_dir in sorted((V1_OUTPUT / "candor_turn_taking").iterdir()):
    if not sample_dir.is_dir():
        continue
    out = read_json(sample_dir / "output.json")
    ann = read_json(sample_dir / "turn_taking.json")[0]
    input_end = ann["timestamp"][0]
    chunks = out["chunks"]
    tor = tor_from_chunks(chunks)
    output_start = chunks[0]["timestamp"][0] if chunks else None
    latency = None if output_start is None else max(0.0, output_start - input_end)
    turn_eval_rows.append({
        "sample_id": sample_dir.name,
        "input_end": input_end,
        "output_start": output_start,
        "output_text": out.get("text", ""),
        "TOR": tor,
        "latency_sec": latency,
    })

turn_eval_df = pd.DataFrame(turn_eval_rows)
display(turn_eval_df)
display(Markdown(f"**Average take turn:** `{turn_eval_df['TOR'].mean() if len(turn_eval_df) else 0.0:.3f}`"))
valid_latencies = turn_eval_df["latency_sec"].dropna()
display(Markdown(f"**Average latency:** `{valid_latencies.mean() if len(valid_latencies) else 0.0:.3f}s`"))


## 11. Evaluation trace: User Interruption

Evaluator gốc dùng OpenAI để chấm semantic rating. Cell này tách phần cấu trúc có thể xem ngay:

- `TOR`: model có trả lời sau interrupt không.
- `latency`: model bắt đầu trả lời sau khi interrupt kết thúc bao lâu.
- `rating`: nếu đã có `rating.json` từ evaluator gốc thì hiển thị; nếu chưa có thì để trống.

In [ ]:
interrupt_eval_rows = []
for sample_dir in sorted((V1_OUTPUT / "synthetic_user_interruption").iterdir()):
    if not sample_dir.is_dir():
        continue
    out = read_json(sample_dir / "output.json")
    ann = read_json(sample_dir / "interrupt.json")[0]
    interrupt_end = ann["timestamp"][1]
    chunks = out["chunks"]
    tor = tor_from_chunks(chunks)
    output_start = chunks[0]["timestamp"][0] if chunks else None
    latency = None if output_start is None else max(0.0, output_start - interrupt_end)
    rating_path = sample_dir / "rating.json"
    rating = read_json(rating_path).get("rating") if rating_path.exists() else None
    interrupt_eval_rows.append({
        "sample_id": sample_dir.name,
        "context": ann["context"],
        "interrupt": ann["interrupt"],
        "interrupt_end": interrupt_end,
        "output_start": output_start,
        "output_text": out.get("text", ""),
        "TOR": tor,
        "latency_sec": latency,
        "rating_if_available": rating,
    })

interrupt_eval_df = pd.DataFrame(interrupt_eval_rows)
display(interrupt_eval_df)
display(Markdown(f"**Average take turn:** `{interrupt_eval_df['TOR'].mean() if len(interrupt_eval_df) else 0.0:.3f}`"))
valid_latencies = interrupt_eval_df["latency_sec"].dropna()
display(Markdown(f"**Average latency:** `{valid_latencies.mean() if len(valid_latencies) else 0.0:.3f}s`"))
if interrupt_eval_df["rating_if_available"].notna().any():
    display(Markdown(f"**Average rating:** `{interrupt_eval_df['rating_if_available'].dropna().mean():.3f}`"))
else:
    display(Markdown("**Average rating:** chưa có `rating.json`; chạy evaluator gốc với OpenAI nếu cần semantic rating."))


## 12. Optional: chạy evaluator gốc

Các cell trên đã trace công thức và kết quả per-sample. Nếu muốn đối chiếu với evaluator gốc của repo, chạy cell này. Lưu ý `user_interruption` cần `OPENAI_API_KEY` để chấm rating semantic.

In [ ]:
import subprocess

RUN_ORIGINAL_EVALUATORS = False
EVAL_SCRIPT = ROOT_DIR / "v1_v1.5" / "evaluation" / "evaluate.py"

if RUN_ORIGINAL_EVALUATORS:
    commands = [
        [sys.executable, str(EVAL_SCRIPT), "--task", "pause_handling", "--root_dir", str(V1_OUTPUT / "synthetic_pause_handling")],
        [sys.executable, str(EVAL_SCRIPT), "--task", "smooth_turn_taking", "--root_dir", str(V1_OUTPUT / "candor_turn_taking")],
    ]
    if os.getenv("OPENAI_API_KEY"):
        commands.append([sys.executable, str(EVAL_SCRIPT), "--task", "user_interruption", "--root_dir", str(V1_OUTPUT / "synthetic_user_interruption")])
    else:
        print("Skip original user_interruption evaluator: missing OPENAI_API_KEY")

    for cmd in commands:
        print("$", " ".join(cmd))
        result = subprocess.run(cmd, text=True, capture_output=True)
        print(result.stdout[-1200:])
        if result.stderr:
            print(result.stderr[-1200:])
else:
    print("RUN_ORIGINAL_EVALUATORS=False; skipped.")
